# Moteur de recherche documentaire

**Programmation de spécialité — Master 1 Informatique**

Interface graphique permettant d'interroger un corpus
de documents Reddit et Arxiv consacré au coronavirus.

Le moteur utilise :
- une représentation Documents × Termes ;
- une pondération TF-IDF ;
- la similarité cosinus ;
- un classement des résultats par pertinence.

Cette interface réutilise les classes Python du projet.

In [ ]:

from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as widgets
from projet.corpus import Corpus
from projet.document import Document
from projet.search_engine import SearchEngine

print("Imports réussis.")

In [ ]:

chemin_corpus = Path("data/corpus.csv")
if not chemin_corpus.is_file():
    raise FileNotFoundError(
        f"Corpus introuvable : {chemin_corpus.resolve()}"
    )
corpus = Corpus.get_instance("Coronavirus")
corpus.load(chemin_corpus)

print("Corpus chargé avec succès.")
print("Nombre de documents :", corpus.get_document_count())

In [ ]:

moteur = SearchEngine(
    corpus,
    show_progress=True
)
print("\nIndexation terminée.")
print(f"Documents : {len(moteur.document_ids)}")
print(f"Mots distincts : {len(moteur.vocab)}")
print(f"Matrice TF : {moteur.mat_TF.shape}")
print(f"Matrice TF-IDF : {moteur.mat_TFxIDF.shape}")

In [ ]:
requetes = [
    "covid vaccine",
    "coronavirus treatment",
    "vaccine research"
]
for requete in requetes:
    print(f"\nRecherche : {requete}")
    resultats = moteur.search(
        mots_cles=requete,
        n=5
    )
    if resultats.empty:
        print("Aucun document trouvé.")
    else:
        display(
            resultats.round({"score": 4})
        )

In [ ]:

titre = widgets.Label(
    value="Moteur de recherche"
)
champ_mots = widgets.Text(
    value="",
    placeholder="Exemple : covid vaccine",
    description="Mots clés :",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px")
)
curseur_nombre = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description="Nombre d'articles à extraire :",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="550px"),
    continuous_update=False
)
bouton_rechercher = widgets.Button(
    description="Rechercher",
    button_style="primary",
    icon="search"
)
zone_resultats = widgets.Output()

In [ ]:

def clique_bouton(bouton) -> None:
    mots_cles: str = champ_mots.value.strip()
    nombre_articles: int = curseur_nombre.value
    with zone_resultats:
        clear_output(wait=True)
        if not mots_cles:
            print("Veuillez saisir des mots-clés.")
            return
        resultats = moteur.search(
            mots_cles=mots_cles,
            n=nombre_articles
        )
        if resultats.empty:
            print("Aucun document pertinent trouvé.")
            return
        print(
            f"{len(resultats)} document(s) trouvé(s) "
            f"pour : {mots_cles}"
        )
        display(
            resultats.round({"score": 4})
        )

In [ ]:

bouton_rechercher.on_click(clique_bouton)
ligne_actions = widgets.HBox([
    curseur_nombre,
    bouton_rechercher
])
interface = widgets.VBox([
    titre,
    champ_mots,
    ligne_actions,
    zone_resultats
])
display(interface)

In [ ]:

bouton_concordancier = widgets.Button(
    description="Concordancier",
    button_style="info"
)
def clique_concordancier(bouton) -> None:
    expression = champ_mots.value.strip()
    with zone_resultats:
        clear_output(wait=True)
        if not expression:
            print("Veuillez saisir un mot ou une expression.")
            return
        concordances = corpus.concorde(
            expression=expression,
            context_size=40
        )
        if concordances.empty:
            print("Aucune concordance trouvée.")
        else:
            print(
                f"{len(concordances)} occurrence(s) trouvée(s)."
            )
            display(concordances.head(20))

bouton_concordancier.on_click(
    clique_concordancier
)
ligne_actions.children = (
    curseur_nombre,
    bouton_rechercher,
    bouton_concordancier
)